# 🚀 [Colab 실습] YOLOv9 실전 — 학습부터 ONNX Export까지

**온디바이스 AI 프로그래밍 · Day 3 「YOLOv9 실시간 파이프라인」 표준 학습 실습**

| 항목 | 내용 |
| --- | --- |
| 위치 | 「YOLO v1 첫걸음」(원리) → **이 실습(진짜 v9 학습·평가·Export)** |
| 도구 | ultralytics (YOLOv9 공식 지원) + PyTorch + ONNX |
| 데이터 | **Aquarium** — 수족관 7클래스 (Roboflow, CC BY 4.0) |
| 환경 | **Google Colab GPU 런타임 (T4)** — 런타임 → 런타임 유형 변경 → T4 GPU |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) · 학습 셀 ☕ 약 15분 1회 |

## 이 실습의 목표

첫걸음에서 손으로 만든 미니 YOLO의 모든 개념 — 그리드·책임·손실 3항·NMS·IoU — 이
**진짜 YOLOv9에서 어떤 모습으로 살아 있는지** 확인하며,
**학습 → 평가(mAP) → 추론 실험 → 테스트셋 시연 → ONNX Export**의 표준 파이프라인을 완주합니다.

오늘의 데이터는 **수족관** — 물고기 떼, 해파리, 펭귄, 상어가 등장합니다. 시각적으로 즐겁지만
만만치 않습니다: 라벨 수가 클래스마다 **최대 25배** 차이 나는 **클래스 불균형**과,
떼 지어 다니는 물고기들의 **밀집 NMS** 문제가 숨어 있습니다. mAP 숫자 뒤의 이 두 현실을
직접 관찰하는 것이 오늘의 숨은 목표입니다.

## 로드맵

| Part | 주제 | 첫걸음·애니메이션 연결 |
| --- | --- | --- |
| 1 | 사전학습 v9 첫 만남 — 출력 텐서 해부 | 4×4 그리드 → 멀티스케일 3549셀 |
| 2 | 학습 — Aquarium 파인튜닝 (☕) | 손실 3항의 실전판 (box·cls·dfl) |
| 3 | 평가 — mAP50 / mAP50-95 해부 | IoU 임계값 + 클래스 불균형의 흔적 |
| 4 | 추론 실험 — conf·NMS 스윕 | NMS 애니메이션 실전판 (물고기 떼) |
| 5 | 테스트셋 시연 — 처음 보는 수조 12장 | 일반화 성능을 눈으로 |
| 6 | ONNX Export — 프레임워크를 넘는 여권 | ONNX 첫걸음·교안 함정 6가지 |

---
# Part 0. 환경 준비 — GPU 확인과 설치

In [ ]:
# GPU 런타임인지 확인 (T4가 보여야 정상 — 안 보이면 런타임 유형을 GPU로!)
!nvidia-smi -L
import torch
DEV = 0 if torch.cuda.is_available() else "cpu"
print("학습 디바이스:", "GPU ✅" if DEV == 0 else "CPU (느립니다 — GPU 런타임 권장!)")

In [ ]:
%pip install -q ultralytics onnx onnxruntime
import ultralytics
ultralytics.checks()

In [ ]:
# 한글 폰트 설정 — 그래프 제목·라벨이 □□로 깨지지 않도록 (시리즈 공통 셀)
import matplotlib.pyplot as plt
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료 ✅")
except Exception as e:
    print("한글 폰트 설정 생략(제목이 깨질 수 있음):", e)
plt.rc("axes", unicode_minus=False)

---
# Part 1. 사전학습 YOLOv9 첫 만남

### Step 1-1. 모델 로드 — GELAN 블록을 눈으로 확인

`yolov9t`(tiny)는 v9 가족의 막내입니다: **2.1M 파라미터, 4.7MB** (논문의 v9-C는 25.3M).
구조를 출력해 애니메이션에서 본 **GELAN**이 실제 블록 이름으로 박혀 있는 것을 확인합시다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov9t.pt")                      # 사전학습(COCO) 가중치 자동 다운로드
n_params = sum(p.numel() for p in model.model.parameters())
print(f"YOLOv9-t: 파라미터 {n_params/1e6:.2f}M")
print()
# 구조에서 GELAN 흔적 찾기
import re
names = {type(m).__name__ for m in model.model.modules()}
gelan = sorted(n for n in names if "ELAN" in n)
print("모델 속 GELAN 계열 블록:", gelan)
print("→ 애니메이션 「YOLOv9 PGI·GELAN」의 그 블록들입니다 (RepNCSPELAN4 = GELAN 본체)")
print()
print("💡 PGI 보조 분기는? — 추론용 배포 가중치에는 이미 '걷어낸' 상태(비계 제거 완료).")
print("   학습 프레임워크 내부에서만 쓰이고, 우리가 받은 모델엔 흔적이 없습니다 — 추론 비용 +0의 증거.")

### Step 1-2. 첫 추론 — 사전학습의 힘

In [ ]:
import matplotlib.pyplot as plt

res = model.predict("https://ultralytics.com/images/bus.jpg",
                    imgsz=416, conf=0.25, verbose=False)[0]
plt.figure(figsize=(7, 8))
plt.imshow(res.plot()[..., ::-1])               # BGR→RGB
plt.axis("off"); plt.title(f"yolov9t 첫 추론 — 검출 {len(res.boxes)}개")
plt.show()

for b in res.boxes:
    print(f"  {res.names[int(b.cls)]:<10} conf {float(b.conf):.2f}  박스 {b.xyxy[0].int().tolist()}")

### Step 1-3. 출력 텐서 해부 — 첫걸음의 4×4가 어떻게 자랐나

원시 출력(NMS 전)의 shape를 직접 확인합니다. 첫걸음과의 대응이 핵심입니다.

In [ ]:
import numpy as np
mdev = next(model.model.parameters()).device      # predict() 후 모델이 GPU에 있을 수 있음
x = torch.zeros(1, 3, 416, 416, device=mdev)      # 입력을 모델과 같은 디바이스에 생성
with torch.no_grad():
    raw = model.model(x)[0].cpu()
print("원시 출력 shape:", tuple(raw.shape), " ← (배치, 4+클래스80, 셀 수)")
print()
print(f"셀 수 3549의 정체: 52² + 26² + 13² = {52**2 + 26**2 + 13**2}")
print(f"  416/8 = 52 (작은 물체 담당 · 촘촘한 그리드)")
print(f"  416/16 = 26 (중간)")
print(f"  416/32 = 13 (큰 물체 담당 · 성긴 그리드)")
print()
print("┌── 첫걸음 미니 YOLO ──────────┬── YOLOv9 ─────────────────────────┐")
print("│ 4×4 그리드 한 벌            │ 52²+26²+13² 멀티스케일 세 벌       │")
print("│ 셀 벡터 [x,y,w,h,conf,p×3]  │ 셀 벡터 [x,y,w,h, cls×80] (앵커프리)│")
print("│ conf 칸 별도                │ conf 없음 — 클래스 점수가 겸임      │")
print("│ 작은 물체의 저주(41%)       │ 52² 그리드가 그 저주의 처방          │")
print("└─────────────────────────────┴───────────────────────────────────┘")

> **✅ Part 1 확인**
> - [ ] GELAN 블록(RepNCSPELAN4)을 모델 구조에서 직접 찾았다
> - [ ] PGI 분기가 배포 가중치에 없는 이유(추론 비용 +0)를 설명할 수 있다
> - [ ] 3549 = 52²+26²+13² 멀티스케일이 "작은 물체의 저주"의 처방임을 이해했다

---
# Part 2. 학습 — Aquarium 파인튜닝

### Step 2-1. 데이터: Aquarium (Roboflow "Aquarium Combined")

미국 수족관 두 곳에서 촬영된 탐지 데이터셋입니다 (출처: Roboflow / Brad Dwyer,
**CC BY 4.0** — 출처 표기 조건으로 자유 사용 가능). Roboflow 원본은 API 키가 필요하므로,
동일 데이터가 포함된 GitHub 공개 미러에서 받습니다 — 아래 셀 하나면 끝납니다.

| 분할 | 이미지 수 | 용도 |
| --- | --- | --- |
| train | 448장 | 학습 |
| valid | 127장 | epoch마다 평가 |
| test | 68장 | **Part 5 시연 — 학습에 전혀 쓰지 않음** (라벨 없는 배경 5장 포함) |

7클래스와 train 라벨 분포 — **불균형에 주목**:

| 클래스 | fish | jellyfish | penguin | shark | puffin | stingray | starfish |
| --- | --- | --- | --- | --- | --- | --- | --- |
| GT 박스 | 1,961 | 385 | 330 | 259 | 175 | 136 | **78** |

> 💡 fish는 starfish의 **25배**입니다. 학습이 끝나면 이 분포가 클래스별 AP에
> 어떤 흔적을 남기는지 Part 3에서 확인합니다 — 실전 데이터의 가장 흔한 함정입니다.

### Step 2-2. 데이터 다운로드

In [ ]:
import urllib.request, zipfile, shutil
from pathlib import Path

DATASET_DIR = Path('/content/datasets/aquarium')

if not (DATASET_DIR / 'train' / 'images').exists():
    zp = Path('/content/aq_repo.zip')
    print('다운로드 중... (약 87MB)')
    urllib.request.urlretrieve(
        'https://codeload.github.com/shivansh65/YOLO_object_detection/zip/refs/heads/main', zp)
    with zipfile.ZipFile(zp) as z:
        root = z.namelist()[0].split('/')[0]
        members = [n for n in z.namelist() if n.startswith(f'{root}/aquarium_dataset/')]
        z.extractall('/content/_aq_tmp', members)
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(f'/content/_aq_tmp/{root}/aquarium_dataset', DATASET_DIR)
    shutil.rmtree('/content/_aq_tmp'); zp.unlink()
    print('완료:', DATASET_DIR)
else:
    print('이미 존재:', DATASET_DIR)

# 학습 설정 yaml 생성 (절대경로)
AQ_YAML = DATASET_DIR / 'aquarium.yaml'
AQ_YAML.write_text(
    f"train: {DATASET_DIR}/train/images\n"
    f"val: {DATASET_DIR}/valid/images\n"
    f"test: {DATASET_DIR}/test/images\n\n"
    "nc: 7\n"
    "names: ['fish', 'jellyfish', 'penguin', 'puffin', 'shark', 'starfish', 'stingray']\n"
)

# 구조 확인 (기대: train 448 / valid 127 / test 68)
ok = True
for split, exp in [('train', 448), ('valid', 127), ('test', 68)]:
    n_img = len(list((DATASET_DIR / split / 'images').glob('*.jpg')))
    n_lbl = len(list((DATASET_DIR / split / 'labels').glob('*.txt')))
    mark = '✅' if n_img == exp else '❌'
    ok &= n_img == exp
    print(f'{mark} {split:>5}: 이미지 {n_img:3d}장 / 라벨 {n_lbl:3d}개 (기대 {exp})')
if not ok:
    print('⚠️ 개수가 다르면 /content/datasets/aquarium 폴더 삭제 후 재실행하세요.')

### Step 2-3. 학습 실행 (☕ T4 기준 약 15분, 1회만)

`imgsz=416`은 이 실습 시리즈의 공통 해상도입니다 — 첫걸음의 텐서 해부(3549셀)와 같은 숫자입니다.

In [ ]:
# ⚠️ 학습 직전 모델을 '새로' 로드합니다 — 중요!
# Part 1의 predict()는 추론 최적화를 위해 모델을 fuse(Conv+BN 합치기)합니다.
# fuse된 모델로 train을 시작하면 가중치 이름이 어긋나 사전학습 이식이
# 150/1339개만 되어(로그로 확인 가능) 사실상 밑바닥 학습이 되어버립니다.
model = YOLO("yolov9t.pt")

results = model.train(
    data=str(AQ_YAML),
    epochs=40,               # 데이터가 작아 넉넉히 (빠른 진행: 20)
    imgsz=416,               # 시리즈 공통 해상도
    batch=16,
    device=DEV,
    name="v9_aquarium",
    plots=True,
)
print("학습 완료! 산출물 폴더:", results.save_dir)

> 🔎 **학습이 제대로 시작됐는지 확인하는 법** — 위 셀 로그 앞부분에서
> `Transferred 1339/1339 items from pretrained weights` 를 찾으세요.
> **1339/1339**여야 정상(사전학습 전체 이식)이고, `150/1339`처럼 나오면 fuse된 모델로
> 학습이 시작된 것입니다(이 경우 mAP가 0 근처에 머뭅니다). 30에폭 후 mAP50이
> 대략 0.5~0.7 수준이면 파인튜닝이 잘 된 것입니다.

### Step 2-4. 학습 곡선 읽기 — 손실 3항의 실전판

첫걸음의 손실 3항(좌표 / conf / 클래스)이 v9에선 이렇게 바뀌었습니다:

| 첫걸음 미니 YOLO | YOLOv9 | 의미 |
| --- | --- | --- |
| 좌표 제곱합 (λ=5) | **box_loss** (CIoU) | 박스를 IoU 기반으로 직접 최적화 |
| — | **dfl_loss** (Distribution Focal) | 박스 경계를 "분포"로 회귀 — 앵커 프리의 크기 학습 무기 |
| conf + 클래스 | **cls_loss** (BCE) | 클래스 점수가 conf 역할까지 겸임 |

In [ ]:
from IPython.display import Image, display
import os
display(Image(os.path.join(results.save_dir, "results.png"), width=980))
print("읽는 법: 왼쪽 3칸(train 손실)과 그 옆(val 손실)이 함께 내려가고,")
print("        오른쪽 mAP50 / mAP50-95가 올라가면 — 3항이 협업 중이라는 뜻입니다.")

> **✅ Part 2 확인**
> - [ ] `Transferred 1339/1339` 로그로 사전학습 이식을 확인했다
> - [ ] box/dfl/cls 3항을 첫걸음의 3항과 대응시킬 수 있다
> - [ ] train 라벨 분포에서 25배 클래스 불균형을 확인했다

---
# Part 3. 평가 — mAP 해부

### Step 3-1. mAP50 vs mAP50-95 — IoU 임계값의 귀환

- **mAP50**: 예측이 정답과 **IoU > 0.5**면 성공으로 치고 계산한 평균 정밀도 — IoU 애니메이션의 그 "합격선"
- **mAP50-95**: 임계값을 0.5, 0.55, ..., 0.95로 **10단계 올려가며** 평균 — 박스가 얼마나 "정확히" 맞는지까지 요구
- 항상 mAP50-95 ≤ mAP50: 자가 빡빡해질수록 점수는 내려갑니다

In [ ]:
best = YOLO(os.path.join(results.save_dir, "weights/best.pt"))
metrics = best.val(data=str(AQ_YAML), imgsz=416, device=DEV, verbose=False)

print(f"mAP50    = {metrics.box.map50:.3f}   (IoU>0.5 합격선)")
print(f"mAP50-95 = {metrics.box.map:.3f}   (0.5~0.95 평균 — 더 빡빡)")
print(f"차이가 곧 '박스 정밀도의 여지'입니다.")
print()
print("클래스별 AP50 (7클래스 전체):")
for name, ap in sorted(zip(metrics.names.values(), metrics.box.ap50), key=lambda x: -x[1]):
    bar = "█" * int(float(ap) * 40)
    print(f"  {name:>10} {float(ap):.3f} {bar}")
print()
print("→ Part 2의 라벨 분포표와 나란히 놓고 보세요. 라벨이 25배 적은 starfish의")
print("  AP는 몇 등인가요? '데이터가 적으면 못 배운다'가 항상 참일까요?")
print("  (반전이 있다면 — 생김새가 독특한 클래스는 적은 데이터로도 잘 배웁니다)")

### Step 3-2. PR 곡선과 혼동행렬 — 그림으로 보는 성적표

In [ ]:
val_dir = metrics.save_dir
display(Image(os.path.join(val_dir, "BoxPR_curve.png"), width=560))
display(Image(os.path.join(val_dir, "confusion_matrix_normalized.png"), width=560))
print("PR 곡선: conf 임계를 훑으며 그린 정밀도-재현율 — 곡선 아래 면적이 AP입니다.")
print("혼동행렬: 어떤 클래스를 어떤 클래스로 착각하는지 — background 열이 미검출입니다.")

> **✅ Part 3 확인**
> - [ ] mAP50과 mAP50-95의 차이를 IoU 임계로 설명할 수 있다
> - [ ] PR 곡선의 면적 = AP 관계를 이해했다
> - [ ] 혼동행렬에서 background(미검출) 열을 읽을 수 있다

---
# Part 4. 추론 파라미터 실험 — conf와 NMS를 실전에서 돌려보기

첫걸음에서 10줄로 구현한 그 두 개의 손잡이가, 실전에선 `predict()`의 인자입니다:
- `conf` — 신뢰도 임계 (낮추면 후보 ↑, 오검출 ↑)
- `iou` — **NMS 억제 임계** (높이면 겹친 박스 관대 → 중복 ↑)

### Step 4-1. conf 스윕

In [ ]:
# 학습에 쓰지 않은 test 분할에서 실험용 이미지 한 장을 고릅니다
from pathlib import Path
test_imgs = sorted(Path("/content/datasets/aquarium/test/images").glob("*.jpg"))
IMG = str(test_imgs[10])
print(f"test 이미지 {len(test_imgs)}장 중 실험 대상: {IMG}")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, c in zip(axes, [0.10, 0.25, 0.60]):
    r = best.predict(IMG, imgsz=416, conf=c, iou=0.5, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(f"conf={c} → {len(r.boxes)}개")
plt.suptitle("conf 임계 스윕 — 낮출수록 후보가 늘고, 오검출도 늘어난다")
plt.tight_layout(); plt.show()

### Step 4-2. NMS(iou) 스윕 — 애니메이션의 ✂를 실전에서

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, t in zip(axes, [0.30, 0.50, 0.90]):
    r = best.predict(IMG, imgsz=416, conf=0.20, iou=t, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(f"NMS iou={t} → {len(r.boxes)}개")
plt.suptitle("NMS 임계 스윕 — 물고기 떼처럼 겹친 곳에서 ✂의 효과가 가장 크다")
plt.tight_layout(); plt.show()
print("💡 첫걸음 NMS 10줄과 「책임 셀·NMS」 애니메이션의 그 IoU>임계 → ✂ 규칙이")
print("   그대로 동작 중입니다. 물고기 떼 사진을 골라(인덱스 변경) 다시 실험해 보세요 —")
print("   진짜로 겹쳐 있는 두 마리와 중복 박스를 NMS가 구분할 수 있을까요?")

> **✅ Part 4 확인**
> - [ ] conf를 낮추면 박스 수가 어떻게 변하는지 관찰했다
> - [ ] NMS iou=0.9에서 중복 박스가 살아남는 것을 확인했다

---
# Part 5. 테스트셋 시연 — 처음 보는 수조 12장

학습(train)에도, epoch별 평가(val)에도 **한 번도 쓰이지 않은 test 68장**에서
12장을 뽑아 내 모델의 실력을 확인합니다. 숫자(mAP)가 아니라 눈으로 보는 일반화 성능입니다.

관찰 포인트:
- **다수 vs 소수 클래스**: fish는 잘 잡고 starfish·puffin은 놓치는 패턴이 보이는가?
- **물고기 떼**: 겹친 개체들에서 박스가 하나로 뭉개지거나 빠지는가? (NMS의 딜레마)
- **배경 이미지의 함정**: test에는 **아무것도 없는 배경 5장**이 섞여 있습니다 —
  모델이 유령 물고기를 만들어내는지(오검출) 지켜보세요.

In [ ]:
import random
random.seed(7)
samples = random.sample(test_imgs, 12)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, img_path in zip(axes.flat, samples):
    r = best.predict(str(img_path), imgsz=416, conf=0.4, iou=0.5, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    names = [r.names[int(c)] for c in r.boxes.cls]
    from collections import Counter as _C
    summary = ", ".join(f"{n}×{k}" if k > 1 else n for n, k in _C(names).items())
    ax.set_title(summary if summary else "(미검출)", fontsize=10)
plt.suptitle("test 12장 시연 — 학습에 전혀 쓰이지 않은 수조 (conf=0.4)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# test 68장 전수 추론 통계 (작아서 수십 초면 끝납니다)
from collections import Counter

counter = Counter()
undetected = 0
bg_false_positives = []
label_dir = Path("/content/datasets/aquarium/test/labels")
for img_path in test_imgs:
    r = best.predict(str(img_path), imgsz=416, conf=0.4, iou=0.5, verbose=False)[0]
    if len(r.boxes) == 0:
        undetected += 1
    counter.update(r.names[int(c)] for c in r.boxes.cls)
    # 라벨 파일이 없는 배경 이미지에서 뭔가 탐지되면 = 확실한 오검출
    if not (label_dir / (img_path.stem + ".txt")).exists() and len(r.boxes) > 0:
        bg_false_positives.append((img_path.name, len(r.boxes)))

print(f"test {len(test_imgs)}장 전체 추론 완료 (conf=0.4)")
print(f"미검출 이미지: {undetected}장")
print("클래스별 탐지 박스 수:")
for name, n in counter.most_common():
    print(f"  {name:>10}: {n}")
print()
if bg_false_positives:
    print(f"👻 배경 이미지에서의 오검출: {bg_false_positives}")
    print("   → 아무것도 없는 사진에서 만들어낸 '유령'입니다. conf를 올리면 사라질까요?")
else:
    print("👻 배경 이미지 5장에서 오검출 없음 — conf=0.4 기준으로는 유령이 없습니다.")
print()
print("→ Part 3의 클래스별 AP50 순위, Part 2의 라벨 분포와 삼각 비교해 보세요.")

> **✅ Part 5 확인**
> - [ ] 다수 클래스(fish)와 소수 클래스(starfish 등)의 검출 차이를 눈으로 확인했다
> - [ ] 배경 이미지 5장에서 오검출(유령)이 나오는지 conf를 바꿔가며 실험했다
> - [ ] 클래스별 탐지 수 · AP50 순위 · train 라벨 분포를 삼각 비교했다

---
# Part 6. ONNX Export — 프레임워크를 넘는 여권

「ONNX Export 첫걸음」에서 밑바닥부터 만든 그 과정 — **트레이스 → 설계도 → 검증** — 을
진짜 도구로 실행합니다. ONNX는 PyTorch를 벗어나 어떤 추론 환경에서든(서버, 모바일, 임베디드)
모델을 실행할 수 있게 하는 표준 교환 형식입니다. 교안 체크리스트 그대로:

| 교안 체크 | export 인자 |
| --- | --- |
| **고정 shape** (dynamic_axes 금지) | `dynamic=False`, `imgsz=416` |
| **opset ≥ 13** | `opset=13` |
| **단순화** (Reshape 난무 청소) | `simplify=True` (onnxslim) |

### Step 6-1. Export 실행

In [ ]:
onnx_path = best.export(format="onnx", imgsz=416, opset=13,
                        simplify=True, dynamic=False)
import os
print(f"생성: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f}MB)")

### Step 6-2. 설계도 검사 — 여권 심사

In [ ]:
import onnx
m = onnx.load(onnx_path)
inp = m.graph.input[0]
dims = [d.dim_value for d in inp.type.tensor_type.shape.dim]
ops = {n.op_type for n in m.graph.node}

print(f"입력 shape : {dims}   ← 배치까지 상수 1로 박제 (고정 shape ✔)")
print(f"opset      : {m.opset_import[0].version}   (≥13 ✔)")
print(f"노드 수    : {len(m.graph.node)}")
print(f"연산자 종류: {len(ops)}종 — 예: {sorted(ops)[:8]} ...")
print()
print("💡 첫걸음의 '설계도 딕셔너리'가 protobuf로 저장된 것 — Netron(netron.app)에")
print("   이 파일을 끌어다 놓으면 그래프를 눈으로 볼 수 있습니다.")

### Step 6-3. 국경 반대편 검증 — PyTorch vs onnxruntime

첫걸음의 클라이맥스("파일만 읽어 재실행 → 같은 답")를 실전 규모로 반복합니다.

In [ ]:
import onnxruntime as ort
import numpy as np, torch

x = np.random.rand(1, 3, 416, 416).astype(np.float32)
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
y_onnx = sess.run(None, {sess.get_inputs()[0].name: x})[0]

cpu_model = YOLO(os.path.join(results.save_dir, "weights/best.pt"))
with torch.no_grad():
    y_pt = cpu_model.model(torch.from_numpy(x))[0].numpy()

diff = np.abs(y_pt - y_onnx).max()
print(f"PyTorch 출력 {y_pt.shape}  vs  ONNX 출력 {y_onnx.shape}")
print(f"최대 오차: {diff:.2e} → 일치(atol 1e-3): {np.allclose(y_pt, y_onnx, atol=1e-3)}")
print()
print("✅ 모델이 프레임워크 국경을 넘었습니다. 오차 ~1e-3은 fp32 연산 순서 차이 수준.")
print("   이 .onnx 파일 하나면 서버·모바일·임베디드 어디서든 같은 답을 냅니다.")

> **✅ Part 6 확인**
> - [ ] 체크리스트 3종(고정 shape·opset 13·simplify)을 인자로 지정했다
> - [ ] onnx.load로 입력 [1,3,416,416]과 opset을 직접 확인했다
> - [ ] PyTorch↔ONNX 출력 일치를 수치로 검증했다

---
# 마무리 — 리포트 과제

### 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | mAP50 | mAP50-95 | 관찰 |
| --- | --- | --- | --- | --- |
| 기준 | epochs=40, imgsz=416, yolov9t | | | |
| 실험1 | epochs 20 / 80 비교 | | | |
| 실험2 | imgsz 640으로 학습·평가 | | | |
| 실험3 | `yolov9s`로 교체 | | | |

**분석 질문 (2~3문장씩):**
1. **클래스 불균형의 흔적**: train 라벨 수(fish 1,961 ~ starfish 78)와 클래스별 AP50 순위는
   얼마나 일치하나요? 순위가 어긋나는 클래스가 있다면(예: 라벨은 적은데 AP가 높은),
   그 클래스의 '생김새'로 이유를 설명해 보세요.
2. 실험1에서 epochs를 80으로 늘리면 train 손실과 **val mAP**의 추세가 언제부터
   갈라지나요? 448장짜리 작은 데이터에서 과적합이 어떻게 관측되는지 설명하세요.
3. 실험2(640)와 실험3(yolov9s)을 비교하세요 — **해상도를 키우는 것**과 **모델을 키우는 것**
   중 이 데이터에서 어느 쪽이 가성비가 좋았고, "실시간 20+ FPS 목표"라면 무엇을 고르겠습니까?

### ✏️ 심화 도전 과제 (선택)

1. **불균형 완화 실험**: starfish·stingray 이미지를 복제(oversampling)해 학습 후
   해당 클래스 AP 변화를 관찰 — 데이터 차원의 불균형 대응 첫걸음
2. **ONNX 단독 추론기**: onnxruntime + 첫걸음의 NMS 10줄로, ultralytics 없이
   `test 이미지 → 박스` 전 과정을 직접 구현 (전처리 letterbox 포함)
3. **FP16으로 한 걸음**: `export(format="onnx", half=True)` 후 파일 크기·출력 오차 비교 —
   양자화 첫걸음의 감각으로 관찰

---

수고하셨습니다! 🎉 이제 여러분은 YOLOv9를 **원리(첫걸음) → 학습·평가 → 테스트셋 검증 → ONNX Export**까지
표준 파이프라인 전체로 다룰 수 있고, 클래스 불균형이 mAP에 남기는 흔적을 **자신의 실험표**로 보일 수 있습니다.
오늘 만든 `best.pt`와 `best.onnx`, 그리고 416이라는 숫자는 이후 온디바이스 배포 실습에서 그대로 이어집니다.

*데이터 출처: Aquarium Combined v2 — Roboflow / Brad Dwyer, CC BY 4.0*